# Personal Chef Agent

In this notebook, we will build a **Personal Chef agent** that can analyze an image of ingredients and suggest recipes based on those ingredients. The agent will load the `./resources/fridge.png` image, analyze the image to understand what ingredients in the provided image, and uses web search tool to provide recipe suggestions from those ingredients.

In [25]:
from dotenv import load_dotenv
from pprint import pprint
import os
from pathlib import Path
import base64
from typing import Dict, Any
from tavily import TavilyClient
from langchain.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage, SystemMessage

load_dotenv()

True

In [37]:

def load_image_as_base64(image_path: str) -> str:
    """Load a local PNG image and convert it to base64 encoded string."""
    image_path = Path(image_path)
    
    if not image_path.exists():
        raise FileNotFoundError(f"Image file not found: {image_path}")
    if image_path.suffix.lower() != ".png":
        raise ValueError(f"Expected a .png file, got: {image_path.suffix}")
    
    with open(image_path, "rb") as image_file:
        base64_encoded = base64.b64encode(image_file.read()).decode("utf-8")
    
    return base64_encoded


def create_human_message_with_image(image_path: str, prompt: str = "Describe this image.") -> HumanMessage:
    """Create a LangChain HumanMessage with a base64-encoded image."""
    base64_image = load_image_as_base64(image_path=image_path)
    message = HumanMessage(
        content=[
            { "type": "text", "text": prompt },
            { "type": "image", "base64": base64_image, "mime_type": "image/png" }
        ]
    )
    return message

In [38]:
@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information"""
    tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
    return tavily_client.search(query=query, search_depth="basic", max_results=4, include_answer='basic')

In [ ]:
chef_agent = create_agent(
    name = "personal_chef_agent",
    model = "gpt-5-nano",
    system_prompt = SystemMessage(content="""
You are a personal chef. The user will give you an image. Your job is to describe the ingredients in the image and suggest recipes that can be made with those ingredients.
Using the web search tool, search the web for recipes that can be made with the ingredients they have.
Return recipe suggestions and eventually the recipe instructions to the user, if requested.
"""),
    tools = [web_search],
    checkpointer = InMemorySaver()
)

In [40]:
config = {
    "configurable": {"thread_id": "personal_chef_thread"}
}

image_path = Path("notebooks/module-1/resources/fridge.png")
if not image_path.exists():
    image_path = Path("resources/fridge.png")

response = chef_agent.invoke(
    input={
        "messages": [
            create_human_message_with_image(
                image_path=str(image_path),
                prompt="Describe the ingredients in this image. Then suggest some recipes I can make with them.",
            )
        ]
    },
    config=config
)

## Output

In [41]:
print(response['messages'][-1].content)
pprint(response)

Here’s what I see in the image:

- Raw meats: bacon slices (in one tray), two beef steaks (alongside), and a tray of sausages (hot dog/sausage style) plus a round deli ham.
- Fresh produce: red and yellow bell peppers, cucumbers.
- Fruit: green and red apples, purple grapes.

Recipe ideas you can make with these ingredients

- Pepper steak with bell peppers (uses: beef steaks + bell peppers)
  - Quick, skillet-friendly beef with colorful peppers. Source idea: Delish pepper steak recipe.
  - Link for reference: https://www.delish.com/cooking/recipe-ideas/a25658914/pepper-steak-recipe/

- Sausage and bell pepper skillet (uses: sausages + bell peppers)
  - A simple one-pan lunch/dinner: sauté sausages with bell peppers, add a quick glaze or simple sauce.
  - Link for reference: https://thedefineddish.com/sausage-and-bell-pepper-skillet/

- One-skillet sausage, peppers, and rice (uses: sausages + peppers; optional rice)
  - Hearty, all-in-one meal. You can cook the rice in the same pan or 

# Notes

---

### When to use `@tool`

`@tool` is not for “any reusable function.” It’s for functions you want the model/agent to be able to call as a tool during an agent run.

The practical difference is:

Regular Python function:

- used by your code directly
- not exposed to the model
- no tool schema/tool-call protocol
- no separate tool tracing behavior

`@tool`:

- turns the function into a LangChain tool/runnable
- gives it a name, description, and input schema
- lets an agent choose and call it
- shows up in LangSmith as a tool run
- supports tool validation/callbacks/tool-call wiring

So for our case:

- `load_image_as_base64`
- `create_human_message_with_image`

are just input-preparation helpers. The model should not decide whether to call them. Your notebook should do that before the agent runs. So plain Python functions are the right fit.

When you should use `@tool`

- When the agent needs access to an external capability during reasoning.
- When the model may decide “I should call this now.”
- When you want structured tool input/output and tool tracing.

Examples:

- `web_search(query)`
- `get_weather(city)`
- `lookup_order(order_id)`
- `run_sql(query)`
- `send_email(...)`

When you usually should not use `@tool`

- Formatting helpers
- Parsing helpers
- Path builders
- Image/base64 converters used only before invocation
- Utility functions the model should never choose itself

So “must use” is really:

- Use a tool when you want agent/tool-calling behavior.
- Don’t use a tool for ordinary internal Python logic.